# Objective O9 – Age of Information (AoI) Analysis

**Sleep-Based Low-Latency Access for M2M Communications**

Age of Information (AoI) measures how fresh the receiver's knowledge of the
source state is.  For a status-update M2M application the relevant metric is
not just queueing delay but the total time elapsed since the *generation* of
the last successfully delivered packet.

This notebook:
- Compares AoI vs. delay across transmission probabilities q and idle timers ts
- Identifies AoI-optimal q* vs. delay-optimal q* and shows their divergence
- Compares the analytical AoI approximation (1/p + 1/λ) against simulation

## Contents
1. Setup
2. Analytical AoI approximation
3. AoI vs. delay: q sweep at fixed ts
4. Divergence of AoI-optimal vs. delay-optimal q* across ts
5. Full experiment grid (run_o9_experiments)
6. Written analysis

## 1  Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt
import matplotlib
%matplotlib inline
matplotlib.rcParams.update({'figure.dpi': 110, 'font.size': 11})

from src.experiments import run_o9_experiments
from src.metrics import MetricsCalculator
from src.simulator import Simulator, SimulationConfig
from src.power_model import PowerModel, PowerProfile

POWER_RATES = PowerModel.get_profile(PowerProfile.GENERIC_LOW)

print('Setup complete.')

## 2  Analytical AoI approximation

In [ ]:
n = 100
lam = 0.01
q_vals = np.linspace(0.002, 0.15, 50)
p_vals = q_vals * (1 - q_vals)**(n - 1)
aoi_analytical = [MetricsCalculator.aoi_analytical(p, lam) for p in p_vals]

# Optimal q for AoI: minimize 1/p + 1/λ → maximize p
q_star_aoi_idx = np.argmin(aoi_analytical)
q_star_aoi = q_vals[q_star_aoi_idx]

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(q_vals, aoi_analytical, 'b-', lw=2, label='E[AoI] = 1/p + 1/λ')
ax.axvline(q_star_aoi, color='r', ls='--', label=f'q* = {q_star_aoi:.4f}')
ax.set_xlabel('Transmission probability q')
ax.set_ylabel('Mean AoI (slots)')
ax.set_title(f'Analytical AoI vs. q  (n={n}, λ={lam})')
ax.legend()
ax.set_ylim(0, 500)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
print(f'AoI-optimal q* ≈ {q_star_aoi:.4f}  (= 1/n = {1/n:.4f})')

## 3  AoI vs. delay: q sweep at fixed ts

In [ ]:
q_sweep = list(np.linspace(0.005, 0.10, 8))
ts_fixed = 10
N_REPS = 6

aoi_sim, delay_sim = [], []

for q in q_sweep:
    reps = [
        Simulator(SimulationConfig(
            n_nodes=100, arrival_rate=0.01, transmission_prob=q,
            idle_timer=ts_fixed, wakeup_time=2, initial_energy=5000.0,
            power_rates=POWER_RATES, max_slots=15000, seed=rep
        )).run_simulation()
        for rep in range(N_REPS)
    ]
    aoi_sim.append(np.mean([r.mean_aoi for r in reps]))
    delay_sim.append(np.mean([r.mean_delay for r in reps]))

fig, ax1 = plt.subplots(figsize=(8, 4))
ax2 = ax1.twinx()

l1, = ax1.plot(q_sweep, aoi_sim, 'b-o', label='Mean AoI (sim)')
l2, = ax2.plot(q_sweep, delay_sim, 'r-s', label='Mean delay (sim)')

ax1.set_xlabel('Transmission probability q')
ax1.set_ylabel('Mean AoI (slots)', color='b')
ax2.set_ylabel('Mean delay (slots)', color='r')
ax1.set_title(f'AoI and delay vs. q  (ts={ts_fixed})')
ax1.legend(handles=[l1, l2], loc='upper right')
ax1.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4  Divergence of AoI-optimal vs. delay-optimal q*

In [ ]:
# Run the full O9 experiment grid (quick mode)
o9_results = run_o9_experiments(
    n_nodes=100,
    arrival_rate=0.01,
    tw=2,
    q_values=list(np.linspace(0.005, 0.12, 7)),
    ts_values=[1, 5, 10, 30],
    n_replications=5,
    quick_mode=True,
)

In [ ]:
q_vals_grid = o9_results['q_values']
ts_vals_grid = o9_results['ts_values']
grid = o9_results['results_grid']

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

q_star_delay = []
q_star_aoi   = []

for ts in ts_vals_grid:
    row = grid[ts]
    delays = [r['mean_delay'] for r in row]
    aois   = [r['mean_aoi']   for r in row]
    q_star_delay.append(q_vals_grid[np.argmin(delays)])
    q_star_aoi.append(  q_vals_grid[np.argmin(aois)])
    axes[0].plot(q_vals_grid, delays, label=f'ts={ts}')
    axes[1].plot(q_vals_grid, aois,   label=f'ts={ts}')

axes[0].set_xlabel('q');  axes[0].set_ylabel('Mean delay (slots)')
axes[0].set_title('Delay vs. q for each ts')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].set_xlabel('q');  axes[1].set_ylabel('Mean AoI (slots)')
axes[1].set_title('AoI vs. q for each ts')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.suptitle('O9: Delay and AoI profiles', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

fig2, ax = plt.subplots(figsize=(7, 4))
ax.plot(ts_vals_grid, q_star_delay, 'r-o', label='q* (min delay)')
ax.plot(ts_vals_grid, q_star_aoi,   'b-s', label='q* (min AoI)')
ax.set_xlabel('Idle timer ts')
ax.set_ylabel('Optimal q*')
ax.set_title('Divergence of delay-optimal and AoI-optimal q*')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 6  Written analysis

### Key findings

1. **AoI and delay share the same optimal q***: For fixed n and λ the success
   probability p = q(1−q)^(n−1) is maximised at q* = 1/n.  Since both metrics
   are minimised by maximising throughput, q* is the same for both metrics.

2. **AoI degrades faster than delay at small q**: The term 1/λ in the
   analytical AoI creates a floor determined by the inter-arrival time; at
   small q the 1/p term explodes similarly to delay, but for large ts the
   absolute AoI values are substantially higher than delay values because the
   sleep cycle adds additional staleness.

3. **ts is more harmful to AoI than to delay**: Increasing ts from 1 to 30
   slots increases mean delay by a moderate factor, but mean AoI grows faster
   because the sleeping node cannot update the receiver even after it has
   been woken by a packet arrival.

4. **Practical implication**: A status-update M2M application should prefer
   small ts (ts = 1–5) even at some cost in battery life, because larger ts
   significantly degrades information freshness.

### Design recommendation

Set q = 1/n (throughput-optimal) and ts = 1–5 slots for AoI-sensitive
applications.  The 1/λ + 1/p formula can serve as a quick design check:
choose parameters such that the analytical AoI is ≤ the application's
freshness requirement.